# claudesub

> Spawn a headless Claude child on this session's compacted history

In [ ]:
#| default_exp claudesub

Claude Code's own subagents share their parent's clikernel kernel, and a fork re-reads the parent's whole context on every turn. `claudesub` gives a session a different kind of helper: a `claude -p` child whose history is the parent's transcript rendered as a compact document (llmsurgery's compaction DSL, a fraction of the tokens with the decisions and evidence kept), running in its own process and so its own kernel. The parent launches it from a background Bash call or a Monitor, reads its text as it works, and can resume it with an answer if it stops to ask. This is Claude Code only: it composes `fastclaude.session`, `llmsurgery.ant.prepare_compaction`, and the dojo's completion registry.

In [ ]:
#| export
import json, os, subprocess, sys
from fastcore.utils import *
from fastcore.script import call_parse
from fastclaude.session import *
from fastclaude.core import claude_env
from llmsurgery.ant import prepare_compaction
from llmdojo.dojo import _completions, dojo_version
from llmdojo.tmpl import launch_config

In [ ]:
from fastcore.test import *
import shutil, tempfile

## The parent session

The launching session is named by `CLAUDE_CODE_SESSION_ID` in the Bash tool's environment, which we verified matches the transcript stem in interactive sessions, subagents, and `-p` children alike. When it is unset, or names no transcript, `parent_sid` refuses rather than guessing: compacting the wrong conversation into a child would be a silent error. The child also needs a dojo completion id to skip the round: the newest one minted under the current tooling version.

In [ ]:
#| export
def parent_sid(
    cwd=None, # Project directory; the current directory if None
):
    "The launching session, from `CLAUDE_CODE_SESSION_ID`; an error rather than a guess when it is unset or names no transcript"
    sid = os.environ.get('CLAUDE_CODE_SESSION_ID')
    if not sid: raise RuntimeError('CLAUDE_CODE_SESSION_ID is unset: claudesub runs from inside a Claude Code session')
    if not (sess_dir(cwd)/f'{sid}.jsonl').exists(): raise FileNotFoundError(f'No transcript for session {sid} under {sess_dir(cwd)}')
    return sid

def dojo_cid():
    "The newest clean-round completion id minted under the current dojo version, or None"
    v = dojo_version()
    ids = [(o['t'],k) for k,o in _completions().items() if o.get('v')==v]
    return max(ids)[1] if ids else None

In [ ]:
os.environ['CLAUDE_CODE_SESSION_ID'] = 'not-a-session'
test_fail(parent_sid, contains='not-a-session')
del os.environ['CLAUDE_CODE_SESSION_ID']
test_fail(parent_sid, contains='CLAUDE_CODE_SESSION_ID')

In [ ]:
cid = dojo_cid()
assert cid is None or len(cid)==4
cid

## The child session

The child does not get a copy of the parent's transcript. `prepare_compaction` renders the parent's conversation into the compact document and builds the five records Claude Code writes for a compaction (boundary, summary, and the visible `/compact` exchange); `prep_sub` saves just those as a new session, under the same project: a headless resume does not look beyond its own project directory. Resuming it loads only what follows the boundary, so the child's context is the compact document plus whatever the harness injects live. A synthetic parent shows the shape:

In [ ]:
#| export
def prep_sub(
    sid=None, # Parent session id; `parent_sid()` if None
    cwd=None, # Project directory; the current directory if None
):
    "Write a session holding only the parent's compacted history, returning its id for the child to resume"
    c = prepare_compaction(sid or parent_sid(cwd), cwd or '.')
    return save_sess(list(c.records), cwd=cwd, ts=True)

In [ ]:
mproj = Path(tempfile.mkdtemp())
turns = tool_turn('Which module handles retries?', 'mcp__clikernel__py', dict(code="rg('retry', 'src')"),
    'src/api.py:12:def with_retry(', 'Retries live in src/api.py.', cwd=mproj)
psid = save_sess(turns, cwd=mproj)
csid = prep_sub(psid, mproj)
child = load_sess(csid, mproj)
test_ne(csid, psid)
test_eq(child[0].subtype, 'compact_boundary')
assert child[1].isCompactSummary and 'retry' in rec_txt(child[1])
test_eq({r.sessionId for r in child}, {csid})
[r.type for r in child]

## The protocol and the command

The child's standing instructions ride in `--append-system-prompt`, so they never enter its transcript and are identical for every child: what it is, that its kernel is its own, that nobody will answer a question mid-task, how the parent resumes it, and which dojo id to present. Everything else is the directive. Standing `claude` arguments come from the same `launch_config('claude')` that `claudedojo` reads, and a caller's extra flags follow them.

In [ ]:
#| export
PROTOCOL = """You are a subagent spawned by another Claude Code session with `claudesub`. The history above is that session's compacted transcript: you hold its decisions and evidence, but none of its kernel state. Your clikernel kernel is your own; no other session shares it. No person is watching, so never wait for an answer or ask a question mid-task: if you are blocked, stop and report exactly what you need, and the parent can resume you with `claudesub -r <your session id> '<answer>'`, which restarts your kernel, so finish whatever does not depend on the answer before you stop. Bootstrap with dojo_start({cid!r}). Then do the directive, and end with a report the parent can act on without reading your transcript: what you did, what you found, and anything left undone."""

def sub_cmd(
    directive, # The child's task
    sid, # Session id for the child to resume
    cid=None, # Dojo completion id to hand the child; `dojo_cid()` if None
    extra=(), # Further `claude` arguments, after the standing ones from `launch_config`
):
    "argv for the headless child: resume `sid` with `directive`, streaming events as JSON lines; refuses without a clean-round id, since a child never plays the round"
    cid = cid or dojo_cid()
    if not cid: raise RuntimeError('No clean dojo round is on record for this tooling version: play one in the parent session first')
    return ['claude', '-p', directive, f'--resume={sid}', '--output-format=stream-json', '--verbose',
        '--append-system-prompt', PROTOCOL.format(cid=cid), *launch_config('claude'), *extra]

In [ ]:
argv = sub_cmd('Summarize src/api.py', csid, cid='ab12', extra=['--model=haiku'])
test_eq(argv[:3], ['claude', '-p', 'Summarize src/api.py'])
assert f'--resume={csid}' in argv and argv[-1]=='--model=haiku'
assert "dojo_start('ab12')" in argv[argv.index('--append-system-prompt')+1]
os.environ['LLMDOJO_STATE_DIR'] = tempfile.mkdtemp()
test_fail(lambda: sub_cmd('x', csid), contains='play one in the parent session')
del os.environ['LLMDOJO_STATE_DIR']
argv[3:6]

## Running the child

`run_sub` runs the command and prints each text part the child emits as it arrives, which is the child's own narration at its own rhythm, then a footer naming the child's session, turn count, and cost. Under a background Bash call that output lands in the task's file; under a Monitor each line is an event. The result event comes back for the caller; a child that exits without one is an error, not a silent success.

In [ ]:
#| export
def run_sub(
    argv, # The child command, from `sub_cmd`
    quiet=False, # Print only the final report and footer, not the child's text as it arrives?
    cwd=None, # Directory the child runs in; the current directory if None
):
    "Run the child, streaming its text, and return its result event"
    p = subprocess.Popen(argv, stdout=subprocess.PIPE, text=True, env=claude_env(), cwd=cwd)
    res = None
    for line in p.stdout:
        try: e = json.loads(line)
        except json.JSONDecodeError: continue
        if e.get('type')=='result': res = e
        elif e.get('type')=='assistant' and not quiet:
            for b in e['message'].get('content', []):
                if b.get('type')=='text' and b['text'].strip(): print(b['text'].strip(), flush=True)
    p.wait()
    if res is None: raise RuntimeError(f'claude exited {p.returncode} without a result')
    if quiet: print(res.get('result', ''))
    print(f"[claudesub {res['subtype']}] session {res['session_id']}, {res['num_turns']} turns, ${res.get('total_cost_usd', 0):.2f}", flush=True)
    return res

A live run (real model spend, so `eval: false`) proves the child truly starts from the compacted history: it answers from facts that exist only in the synthetic parent above.

In [ ]:
#| eval: false
res = run_sub(sub_cmd('From the history only: which file holds the retries? Reply in one line.', csid, extra=['--model=haiku']), cwd=mproj)
assert 'api.py' in res['result']

## The command line

`claudesub 'directive'` is the whole parent-side API, and `-r <child-sid> 'answer'` continues a child that stopped to ask, keeping its context. As with `claudedojo`, `nested=True` forwards unrecognized flags to `claude` in `--flag=value` form, so `--model=haiku` or `--max-turns=20` need no wrapper support. A failed child exits non-zero, which the background task notification reports.

In [ ]:
#| export
@call_parse(nested=True, pos=['directive'])
def main(
    directive:str, # The child's task
    Resume:str=None, # A child session id to continue with `directive` as its next prompt, instead of spawning from this session
    sid:bool=False, # Print the prepared child session id instead of launching
    Quiet:bool=False, # Print only the child's final report, not its progress text
    cwd:str=None, # Project directory to work from, for the parent lookup, the child session, and the child itself; the current directory if None
):
    "Spawn a headless Claude child on this session's compacted history and stream its text; unrecognized `--flag=value` args go to `claude`"
    if cwd: os.chdir(cwd)
    s = Resume or prep_sub()
    if sid: return print(s)
    res = run_sub(sub_cmd(directive, s, extra=sys.argv[1:]), quiet=Quiet)
    if res['subtype'] != 'success': sys.exit(1)

## Cleanup

In [ ]:
shutil.rmtree(sess_dir(mproj))
shutil.rmtree(mproj)

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()